GRU LSTM (window size =8)

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import ast
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import mean_squared_error, mean_absolute_error
from keras.models import Sequential
from keras.layers import LSTM, GRU, Dense
from sklearn.preprocessing import MinMaxScaler


def create_dataset(data, look_back=8):
    X, Y = [], []
    for i in range(len(data) - look_back - 1):
        X.append(data[i:(i + look_back), 0])
        Y.append(data[i + look_back, 0])
    return np.array(X), np.array(Y)


with open('/content/PaperReady_numeric_esg.txt', 'r') as f:
    content = f.read()
    start = content.find('{')
    end = content.rfind('}') + 1
    numeric_esg = ast.literal_eval(content[start:end])

company_results = []
look_back = 8

for company, values in tqdm(numeric_esg.items(), desc="Pure GRU+LSTM Model"):
    series = pd.Series(values).dropna()
    if len(series) < 20:
        continue

    split_idx = int(len(series) * 0.75)
    train, test = series[:split_idx], series[split_idx:]

    # Scale the entire series (training + testing)
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_series = scaler.fit_transform(series.values.reshape(-1, 1))

    # Prepare train and test sets after scaling
    train_scaled = scaled_series[:split_idx]
    test_scaled = scaled_series[split_idx:]

    # Create dataset for training
    X_train, Y_train = create_dataset(train_scaled, look_back)
    X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))

    # Build and train the model (GRU + LSTM)
    model = Sequential()
    model.add(GRU(50, input_shape=(look_back, 1), return_sequences=True))
    model.add(LSTM(50))
    model.add(Dense(1))
    model.compile(loss='mean_squared_error', optimizer='adam')
    model.fit(X_train, Y_train, epochs=20, batch_size=1, verbose=0)

    # Forecasting via recursive prediction
    test_inputs = train_scaled[-look_back:].reshape(-1, 1)
    predicted_scaled = []
    for _ in range(len(test)):
        x_input = test_inputs[-look_back:].reshape(1, look_back, 1)
        pred = model.predict(x_input, verbose=0)
        test_inputs = np.append(test_inputs, pred)
        predicted_scaled.append(pred[0][0])

    # Inverse scale predictions
    predicted = scaler.inverse_transform(np.array(predicted_scaled).reshape(-1, 1)).flatten()
    actual = test.values

    # Calculate Metrics
    mse = mean_squared_error(actual, predicted)
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mse)
    error_rate = np.mean(np.abs(actual - predicted) / np.where(actual == 0, np.nan, np.abs(actual))) * 100
    mape = np.mean(np.abs((actual - predicted) / np.where(actual == 0, np.nan, actual))) * 100
    variance = np.var(actual)
    wql = mse / variance if variance != 0 else np.nan

    # Theil's U
    theils_u = np.sqrt(np.sum((predicted - actual) ** 2)) / (
        np.sqrt(np.sum(predicted ** 2)) + np.sqrt(np.sum(actual ** 2))
    )

    # Bias
    bias = np.mean(predicted - actual)

    company_results.append({
        "Company": company,
        "MSE": round(mse, 4),
        "MAE": round(mae, 4),
        "RMSE": round(rmse, 4),
        "Error Rate (%)": round(error_rate, 4),
        "MAPE (%)": round(mape, 4),
        "WQL": round(wql, 4),
        "Theil's U": round(theils_u, 4),
        "Bias": round(bias, 4),
        "Actual": actual.tolist(),
        "Predicted": predicted.tolist()
    })

# Final Results DataFrame
results_df = pd.DataFrame(company_results)

print("\nPure GRU + LSTM Model Performance:")
metrics = ["MSE", "MAE", "Error Rate (%)", "MAPE (%)", "WQL", "Theil's U", "Bias"]
avg_metrics = {}

for metric in metrics:
    avg = np.nanmean(results_df[metric])  # Ignore NaNs
    avg_metrics[metric] = avg
    print(f"Average {metric}: {round(avg, 4)}")

# Average RMSE from avg MSE
if not np.isnan(avg_metrics["MSE"]):
    avg_rmse = np.sqrt(avg_metrics["MSE"])
else:
    avg_rmse = np.nan
print(f"Average RMSE (sqrt of avg MSE): {round(avg_rmse, 4)}")

# Save results
results_df.to_excel("pure_gru_lstm_results_window8.xlsx", index=False)
print("\n✅ Results saved to 'pure_gru_lstm_results_window8.xlsx'")


Pure GRU+LSTM Model: 100%|██████████| 495/495 [1:04:42<00:00,  7.84s/it]



Pure GRU + LSTM Model Performance:
Average MSE: 1071.3619
Average MAE: 29.8447
Average Error Rate (%): 162.8933
Average MAPE (%): 162.8933
Average WQL: 7.65206878072668e+26
Average Theil's U: 0.4058
Average Bias: 29.2801
Average RMSE (sqrt of avg MSE): 32.7317

✅ Results saved to 'pure_gru_lstm_results_window8.xlsx'
